In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

options = Options()
# Argumentos de estabilidad para Docker
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")

try:
# El Manager instala el driver compatible con la versi n de Chrome instalada
    driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
    )

    driver.get("https://www.google.com")
    print(f" CONECTADO ! T t u l o de la p g i n a: {driver.title}")
    driver.quit()

except Exception as e:
    print(f" Error: {e}")

 CONECTADO ! T t u l o de la p g i n a: Google


In [11]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

# ==========================================================
# 1. MOTOR DE NAVEGACIÓN (Configuración de Laboratorio)
# ==========================================================
def iniciar_navegador():
    options = Options()
    options.binary_location = "/usr/bin/google-chrome"
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    )

    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )


# ==========================================================
# 2. CONFIGURACIÓN DEL GRUPO (Cada grupo modifica aquí)
# ==========================================================

# Pegar la URL del sitio asignado (Retail, Energía, E-commerce, etc.)
URL_OBJETIVO = "https://books.toscrape.com/"

# Definir las etiquetas técnicas (clases CSS) encontradas con el Inspector de Chrome
SELECTOR_CONTENEDOR = "article.product_pod"
SELECTOR_DATO_A = "h3 a"
SELECTOR_DATO_B = ".product_price"


# ==========================================================
# 3. EJECUCIÓN DEL SCRAPING
# ==========================================================
driver = iniciar_navegador()
dataset_final = []

try:
    print(f" Conectando a la fuente de datos...")
    driver.get("https://books.toscrape.com")
    time.sleep(5)  # Tiempo de espera para carga de datos dinámicos

    # Capturamos todos los bloques de información
    elementos = driver.find_elements(By.CSS_SELECTOR, SELECTOR_CONTENEDOR)
    print(f" Se encontraron {len(elementos)} registros potenciales.")

    for item in elementos:
        try:
            # Extracción genérica de datos
            columna_a = item.find_element(By.CSS_SELECTOR, SELECTOR_DATO_A).text
            columna_b = item.find_element(By.CSS_SELECTOR, SELECTOR_DATO_B).text

            # Limpieza básica de caracteres no numéricos (opcional según el proyecto)
            valor_limpio = "".join(filter(str.isdigit, columna_b))

            dataset_final.append({
                "variable_nombre": columna_a,
                "variable_valor": valor_limpio,
                "status": 1.0  # Indicador de registro procesado (float)
            })

        except:
            # Si un elemento está vacío o es un anuncio, saltar al siguiente
            continue

    # ==========================================================
    # 4. SALIDA DE DATOS (Para análisis posterior)
    # ==========================================================
    if dataset_final:
        df = pd.DataFrame(dataset_final)
        nombre_archivo = "datos_extraidos_grupo.csv"
        df.to_csv(nombre_archivo, index=False)
        print(f" Proceso exitoso. Archivo generado: {nombre_archivo}")
        print(df.head())  # Muestra de los primeros 5 datos
    else:
        print(" No se capturaron datos. Revisar los selectores CSS.")

except Exception as e:
    print(f" Error durante la ejecución: {e}")

finally:
    driver.quit()
    print(" Navegador cerrado.")

 Conectando a la fuente de datos...
 Se encontraron 20 registros potenciales.
 Proceso exitoso. Archivo generado: datos_extraidos_grupo.csv
                variable_nombre variable_valor  status
0            A Light in the ...           5177     1.0
1            Tipping the Velvet           5374     1.0
2                    Soumission           5010     1.0
3                 Sharp Objects           4782     1.0
4  Sapiens: A Brief History ...           5423     1.0
 Navegador cerrado.


In [12]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

# --- PASO 1: CONFIGURACIÓN DEL DRIVER (EL CUERPO) --
# Se define al principio para evitar abrir múltiples navegadores innecesariamente
options = Options()
options.add_argument("--headless")  # Ejecución invisible en el servidor Docker
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=options)

# --- PASO 2: INICIALIZACIÓN DE VARIABLES (LA MEMORIA) --
# La lista debe estar FUERA del bucle para acumular los datos de todas las páginas
datos_totales = []
paginas_objetivo = 5  # Límite para el ejercicio de laboratorio

try:
    # URL inicial del caso de estudio
    driver.get("https://books.toscrape.com/")

    # --- PASO 3: BUCLE DE PAGINACIÓN --
    for i in range(paginas_objetivo):
        print(f"--- Procesando Página {i+1} ---")

        # Espera explícita para asegurar que los elementos carguen
        WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.CLASS_NAME, "product_pod"))
        )

        # CAPTURA: Extraer elementos de la vista actual
        productos = driver.find_elements(By.CLASS_NAME, "product_pod")
        for p in productos:
            datos_totales.append(p.text)

        # NAVEGACIÓN: Lógica para saltar a la siguiente página
        try:
            # Buscamos el botón "Siguiente" usando XPATH
            boton_siguiente = driver.find_element(
                By.XPATH,
                #"//span[contains(text(),'Siguiente')]|//a[@title='Siguiente']"
                boton_siguiente = driver.find_element(By.XPATH, "//a[contains(text(),'next')]")
            )

            # Hacemos clic y esperamos la carga
            boton_siguiente.click()
            time.sleep(3)

        except Exception:
            print("Se ha llegado a la última página o el botón no es visible.")
            break

    print("\nProceso finalizado con éxito.")
    print(f"Total de registros capturados: {len(datos_totales)}")

except Exception as e:
    print(f"Error detectado durante la ejecución: {e}")

finally:
    # --- PASO 4: CIERRE SEGURO --
    driver.quit()

--- Procesando Página 1 ---
Se ha llegado a la última página o el botón no es visible.

Proceso finalizado con éxito.
Total de registros capturados: 20
